## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### Cartesian Product (`cross`) and Named Compound Sets
#### AMPL Formulation

In [3]:
%%ampl_eval
reset;              # clear previous model/data

# Transportation model — naming the Cartesian product set
set ORIG;                           # origin cities
set DEST;                           # destination cities
set LINKS = ORIG cross DEST;        # all (origin, destination) pairs

param supply {ORIG} >= 0;           # supply available at each origin
param demand {DEST} >= 0;           # demand required at each destination
param cost   {LINKS} >= 0;          # cost per unit shipped on each link

# feasibility check: total supply must equal total demand
check: sum {i in ORIG} supply[i] = sum {j in DEST} demand[j];

var Trans {LINKS} >= 0;             # decision: shipment quantity on each link

minimize TotalCost:                 # objective: minimize total transportation cost
    sum {(i,j) in LINKS} cost[i,j] * Trans[i,j];

subject to Supply {i in ORIG}:      # supply constraints: all supply must be shipped
    sum {j in DEST} Trans[i,j] = supply[i];

subject to Demand {j in DEST}:      # demand constraints: all demand must be satisfied
    sum {i in ORIG} Trans[i,j] = demand[j];

In [ ]:
%%ampl_eval
data;

# Origin and destination sets
set ORIG := O1 O2 O3;
set DEST := D1 D2 D3;

# Supply at origins
param supply := O1 20, O2 30, O3 25;

# Demand at destinations (balanced: total = 75)
param demand := D1 25, D2 25, D3 25;

# Transportation cost per (origin, destination)
param cost:
       D1  D2  D3 :=
O1     2   4   5
O2     3   1   7
O3     6   2   3;

option solver highs;               # choose HiGHS linear solver
solve;                             # solve the optimization model
display Trans;                     # show optimal Trans quantities          

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 155
5 simplex iterations
0 barrier iterations
Trans :=
O1 D1   20
O1 D2    0
O1 D3    0
O2 D1    5
O2 D2   25
O2 D3    0
O3 D1    0
O3 D2    0
O3 D3   25
;



#### amplpy Formulation

In [6]:
# Sets
ampl.set['ORIG'] = ['O1', 'O2', 'O3']
ampl.set['DEST'] = ['D1', 'D2', 'D3']

# Supply
ampl.param['supply'].set_values({
    'O1': 20,
    'O2': 30,
    'O3': 25})

# Demand
ampl.param['demand'].set_values({
    'D1': 25,
    'D2': 25,
    'D3': 25})

import pandas as pd

# Cost matrix (ORIG x DEST)
cost = pd.DataFrame({
    'D1': [2, 3, 6],
    'D2': [4, 1, 2],
    'D3': [5, 7, 3]
}, index=['O1', 'O2', 'O3'])

ampl.param['cost'].set_values(cost)

ampl.set_option('solver', 'highs')  # Choose HiGHS solver
ampl.solve()                        # Solve the optimization model

# Display optimal transportation plan (Trans variable)
trans = ampl.var['Trans'].get_values()
print(trans)

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 155
0 simplex iterations
0 barrier iterations
   index0       index1    |  Trans.val  
    'O1'           'D1'     |      20     
    'O1'           'D2'     |      0      
    'O1'           'D3'     |      0      
    'O2'           'D1'     |      5      
    'O2'           'D2'     |      25     
    'O2'           'D3'     |      0      
    'O3'           'D1'     |      0      
    'O3'           'D2'     |      0      
    'O3'           'D3'     |      25     

